In [1]:
# STdeconvolve Pipeline (Fixed HVGs & Ks)

# Load libraries
suppressPackageStartupMessages({
  library(Seurat)
  library(STdeconvolve)
  library(Matrix)
  library(dplyr)
  library(tidyr)
  library(pheatmap)
  library(viridis)
})

set.seed(42)

synth_sp_path = "/Users/romanperikzavodskii/Desktop/4. Ery/CoDeconv/Breast Cancer/test_bench/Synthetic Visium/filtered_feature_bc_matrix.h5"
real_sp_path = '/Users/romanperikzavodskii/Desktop/4. Ery/CoDeconv/Breast Cancer/Visium/Breast Cancer/filtered_feature_bc_matrix.h5'


# 1. Load Reference
cat("Loading SC Reference for Correlation...\n")
sc_obj <- readRDS("Seurat_Clusters.rds")
Idents(sc_obj) <- sc_obj$CellType
sc_avg_expr <- as.matrix(AverageExpression(sc_obj, return.seurat = FALSE)$RNA)

# 2. Phase 1 Function
run_stdeconv_phase1 <- function(sp_path, hvg_file, k, prefix) {
  cat(sprintf("\nPhase 1: LDA & Heatmap for %s (K=%d)\n", prefix, k))
  
  sp_counts <- Read10X_h5(sp_path)
  hvgs_list <- read.csv(hvg_file, header = FALSE)[, 1]
  shared_hvgs <- intersect(rownames(sp_counts), hvgs_list)
  
  sp_counts_subset <- sp_counts[shared_hvgs, ]
  spot_sums <- Matrix::colSums(sp_counts_subset)
  sp_counts_subset <- sp_counts_subset[, spot_sums > 0]
  
  # Build native corpus
  corpus_matrix <- t(as.matrix(sp_counts_subset))
  
  # Fit LDA
  cat("Fitting LDA model...\n")
  ldas <- fitLDA(corpus_matrix, Ks = c(k), plot=FALSE, verbose=FALSE)
  optLDA <- optimalModel(models = ldas, opt = "min")
  
  # Get Theta (Proportions) and Beta (Gene Weights)
  res <- getBetaTheta(optLDA, corpus_matrix)
  
  # Correlate Topics with Reference Cell Types
  cat("Calculating Pearson correlation for Heatmap...\n")
  shared_genes <- intersect(colnames(res$beta), rownames(sc_avg_expr))
  cor_mat <- cor(t(res$beta[, shared_genes]), sc_avg_expr[shared_genes, ], method = "pearson")
  
  # Plot and Save Heatmap
  png_filename <- paste0(prefix, "_Correlation.png")
  pheatmap(cor_mat, 
           color = viridis(100, option = "plasma"), 
           display_numbers = round(cor_mat, 2), # Shows the actual correlation values on the map!
           number_color = "white",
           cluster_rows = TRUE, 
           cluster_cols = TRUE,
           main = paste("Topic to Cell-Type Correlation:", prefix),
           filename = png_filename,
           width = 10, height = 7)
           
  cat(sprintf("Saved heatmap to %s. Please inspect it to create your mapping!\n", png_filename))
  
  # Return Theta + Beta + spot UMI totals (in HVG space) for Phase 2
  spot_umi_hvg <- Matrix::rowSums(corpus_matrix)
  return(list(theta = res$theta, beta = res$beta, spot_umi = spot_umi_hvg))
}

# 3. Execute Phase 1
phase1_synth <- run_stdeconv_phase1(synth_sp_path, "fixed_hvgs.csv", 12, "stdeconv_synth")
phase1_real  <- run_stdeconv_phase1(real_sp_path, "fixed_hvgs_real.csv", 11, "stdeconv_real")

Loading SC Reference for Correlation...


As of Seurat v5, we recommend using AggregateExpression to perform pseudo-bulk analysis.
This message is displayed once per session.



Phase 1: LDA & Heatmap for stdeconv_synth (K=12)
Fitting LDA model...


Filtering out cell-types in pixels that contribute less than 0.05 of the pixel proportion.




Calculating Pearson correlation for Heatmap...
Saved heatmap to stdeconv_synth_Correlation.png. Please inspect it to create your mapping!

Phase 1: LDA & Heatmap for stdeconv_real (K=11)
Fitting LDA model...


Filtering out cell-types in pixels that contribute less than 0.05 of the pixel proportion.




Calculating Pearson correlation for Heatmap...
Saved heatmap to stdeconv_real_Correlation.png. Please inspect it to create your mapping!


In [2]:
# Topic Mappings

map_synth <- c(
  "1"  = "TAMs",
  "2"  = "Tumor",
  "3"  = "Dividing Tumor",
  "4"  = "Dividing Tumor",
  "5"  = "Myoepithelial Cells",
  "6"  = "Plasma Cells",
  "7"  = "Luminal B",
  "8"  = "Fibroblasts",
  "9"  = "Endothelial Cells",
  "10" = "T-cells",
  "11" = "Luminal A",
  "12" = "Tumor"
)

map_real <- c(
  "1"  = "Tumor",
  "2"  = "Dividing Tumor",
  "3"  = "Luminal A",
  "4"  = "Luminal B",
  "5"  = "Myoepithelial Cells",
  "6"  = "Tumor",
  "7"  = "T-cells",
  "8"  = "Luminal A",
  "9"  = "Plasma Cells",
  "10" = "Fibroblasts",
  "11" = "Tumor"
)

In [3]:
# Phase 2 Function
# REPLACES the old theta x reference reconstruction with the real
# STdeconvolve beta-derived per-CellType GEX (UMI-scaled).
run_stdeconv_phase2 <- function(phase1_res, map, prefix) {

  theta_matrix <- phase1_res$theta              # spot x topic
  beta_matrix  <- phase1_res$beta               # topic x gene (HVG space, rows sum ~1)
  spot_umi     <- phase1_res$spot_umi           # named numeric, per-spot UMI totals (HVG space)

  # 1. Proportions (unchanged logic)
  props <- as.data.frame(theta_matrix)

  missing <- setdiff(colnames(props), names(map))
  if (length(missing) > 0) {
    stop(paste("ERROR: You forgot to map these topics:", paste(missing, collapse=", ")))
  }
  colnames(props) <- map[colnames(props)]

  props$Spot <- rownames(props)
  props_final <- props %>%
    pivot_longer(-Spot, names_to = "CellType", values_to = "Prop") %>%
    group_by(Spot, CellType) %>%
    summarise(Prop = sum(Prop), .groups = "drop") %>%
    pivot_wider(names_from = "CellType", values_from = "Prop", values_fill = 0) %>%
    as.data.frame()

  rownames(props_final) <- props_final$Spot
  props_final$Spot <- NULL

  write.csv(props_final, paste0(prefix, "_props.csv"), row.names = TRUE)
  cat(sprintf("Saved proportions to %s_props.csv\n", prefix))

  # 2. Per-CellType GEX from STdeconvolve's own beta
  # Per-topic UMI mass: sum_over_spots( theta[spot,k] * spot_umi[spot] )
  # Topic gene pseudobulk: topic_mass[k] * beta[k, :]    (UMI scale, HVGs only)
  cat("Building per-CellType GEX from LDA beta...\n")

  # Align spots between theta and spot_umi
  common_spots <- intersect(rownames(theta_matrix), names(spot_umi))
  theta_aligned <- theta_matrix[common_spots, , drop = FALSE]
  umi_aligned   <- spot_umi[common_spots]

  # topic_mass: named vector length = n_topics
  topic_mass <- colSums(theta_aligned * umi_aligned)  # spot-wise multiply broadcasts
  names(topic_mass) <- colnames(theta_matrix)

  # Scale beta rows by topic mass: gene-major UMI matrix (genes x topics)
  topic_gex <- t(beta_matrix) %*% diag(topic_mass)
  colnames(topic_gex) <- colnames(theta_matrix)
  rownames(topic_gex) <- colnames(beta_matrix)

  # Aggregate topics -> CellTypes (sum columns mapped to the same CellType)
  ct_for_topic <- map[colnames(topic_gex)]
  unique_cts <- unique(ct_for_topic)
  ct_gex <- sapply(unique_cts, function(ct) {
    cols <- which(ct_for_topic == ct)
    rowSums(topic_gex[, cols, drop = FALSE])
  })
  rownames(ct_gex) <- rownames(topic_gex)
  colnames(ct_gex) <- unique_cts

  # Save per-CellType GEX (genes x CellType, UMI-scaled, HVG-restricted)
  write.csv(ct_gex, paste0(prefix, "_celltype_gex.csv"), row.names = TRUE)
  cat(sprintf("Saved per-CellType GEX to %s_celltype_gex.csv  (%d genes x %d CellTypes)\n",
              prefix, nrow(ct_gex), ncol(ct_gex)))

  return(list(props = props_final, ct_gex = ct_gex, topic_mass = topic_mass))
}

# Execute Phase 2
res_synth <- run_stdeconv_phase2(phase1_synth, map_synth, "stdeconv_synth")
res_real  <- run_stdeconv_phase2(phase1_real,  map_real,  "stdeconv_real")


Saved proportions to stdeconv_synth_props.csv
Building per-CellType GEX from LDA beta...
Saved per-CellType GEX to stdeconv_synth_celltype_gex.csv  (3000 genes x 10 CellTypes)
Saved proportions to stdeconv_real_props.csv
Building per-CellType GEX from LDA beta...
Saved per-CellType GEX to stdeconv_real_celltype_gex.csv  (3000 genes x 8 CellTypes)
